# Step 2: Specialized Embeddings (Metapath2Vec)

In the previous notebook, we used **Node2Vec** to generate embeddings. However, Node2Vec is "blind" to node types. In this notebook, we implement **Metapath2Vec**, which is designed specifically for graphs with different types of nodes (Heterogeneous Graphs).

## What is a Metapath?
A metapath is a predefined sequence of node types that a random walk must follow. For our recommendation system, we use the **UMU (User-Movie-User)** metapath.

### Why UMU?
1. **Identity Awareness:** It ensures the walker doesn't get "stuck" jumping between users or between movies. It forces a jump from a user to a movie and back.
2. **Collaborative Filtering Logic:** By walking `User A -> Movie X -> User B`, we are explicitly telling the model that User A and User B are related *because* they shared a movie. 
3. **Semantic Consistency:** It ensures that the "context window" in our Word2Vec model always contains a balanced mix of users and movies, leading to more accurate link predictions.

## Step 1: Setup and Data Loading

We start by rebuilding our bipartite graph. Remember our prefix convention: `u_` for users and `m_` for movies.

In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import random
from gensim.models import Word2Vec
import matplotlib.pyplot as plt

# Load MovieLens 100k data
r_cols = ['user_id', 'movie_id', 'rating', 'unix_timestamp']
ratings = pd.read_csv('data/u.data', sep='\t', names=r_cols, encoding='latin-1')

# Build Graph (Ratings >= 4 are positive links)
G = nx.Graph()
positive_ratings = ratings[ratings['rating'] >= 4]

for _, row in positive_ratings.iterrows():
    G.add_edge(f"u_{row['user_id']}", f"m_{row['movie_id']}")

print(f"Nodes: {G.number_of_nodes()} | Edges: {G.number_of_edges()}")

Nodes: 2389 | Edges: 55375


## Step 2: Implementing the Metapath Random Walk (UMU)

Unlike Node2Vec, where we pick any neighbor, here we must pick a neighbor of a **different type**. 

If we are at a **User node**, the next step **must** be a **Movie node**.
If we are at a **Movie node**, the next step **must** be a **User node**.

This creates a structured sequence: `U -> M -> U -> M -> U...`

In [2]:
def get_metapath_walks(graph, walk_length, num_walks):
    """
    Generates walks following the User-Movie-User (UMU) metapath.
    """
    walks = []
    nodes = list(graph.nodes())
    
    for _ in range(num_walks):
        random.shuffle(nodes)
        for node in nodes:
            walk = [node]
            while len(walk) < walk_length:
                curr = walk[-1]
                neighbors = list(graph.neighbors(curr))
                
                if len(neighbors) == 0:
                    break
                
                # In a bipartite graph, the neighbor of a 'u_' is always 'm_'
                # and vice-versa. So standard random choice on a bipartite 
                # graph automatically follows the U-M-U-M metapath!
                next_node = random.choice(neighbors)
                walk.append(next_node)
                
            walks.append(walk)
    return walks

# Generate Structured Walks
WALK_LENGTH = 10
NUM_WALKS = 80

print("Generating U-M-U-M structured walks...")
metapath_walks = get_metapath_walks(G, WALK_LENGTH, NUM_WALKS)
print(f"Generated {len(metapath_walks)} walks.")
print(f"Sample walk: {metapath_walks[0]}")

Generating U-M-U-M structured walks...
Generated 191120 walks.
Sample walk: ['m_225', 'u_330', 'm_989', 'u_330', 'm_193', 'u_455', 'm_778', 'u_477', 'm_553', 'u_796']


## Step 3: Training Metapath2Vec

We train our Word2Vec model on these structured walks. Because the walks are strictly `U-M-U-M`, the model learns to associate Users with the Movies they like, and with the Users who share those movies, with much higher precision.

In [3]:
print("Training Metapath2Vec model...")
model = Word2Vec(
    metapath_walks, 
    vector_size=128, # Increased dimension for better resolution
    window=5, 
    min_count=1, 
    sg=1, # Skip-gram
    workers=4, 
    epochs=10 # More epochs for structured learning
)
print("Training complete.")

Training Metapath2Vec model...
Training complete.


## Step 4: Comparing with Node2Vec (Identity Check)

In standard Node2Vec, a User vector might be very close to another User vector. 
In Metapath2Vec, we want to ensure that a **User vector** is being pulled toward the **Movies they actually like**.

In [4]:
def recommend_metapath(user_id, top_n=5):
    user_node = f"u_{user_id}"
    if user_node not in model.wv:
        return "User not found."
    
    already_liked = set(G.neighbors(user_node))
    
    # Get top 100 similar nodes
    similar = model.wv.most_similar(user_node, topn=100)
    
    recs = []
    for node, score in similar:
        # Logic: We only recommend Movies (m_) that the user hasn't seen
        if node.startswith('m_') and node not in already_liked:
            movie_id = node.replace('m_', '')
            recs.append((movie_id, score))
        if len(recs) >= top_n:
            break
    return recs

test_user = 1
recs = recommend_metapath(test_user)
print(f"Metapath2Vec Recommendations for User {test_user}:")
for mid, score in recs:
    print(f"Movie ID: {mid} | Score: {score:.4f}")

Metapath2Vec Recommendations for User 1:
Movie ID: 1241 | Score: 0.3465
Movie ID: 1323 | Score: 0.3370
Movie ID: 1555 | Score: 0.3310
Movie ID: 1529 | Score: 0.3285
Movie ID: 1475 | Score: 0.3272


## 💡 Key Takeaway: Why this worked

In a **Bipartite Graph**, any walk *automatically* alternates between node types. 
- From User, you can only go to Movie.
- From Movie, you can only go to User.

By simply performing random walks on our structured graph, we have effectively implemented the **UMU metapath**. The "Magic" of Metapath2Vec in this case comes from the **skip-gram training** on these perfectly alternating sequences, which forces the neural network to learn the relationship between a user and their multi-hop movie connections.